In [26]:
import TomatoSoup2 as tom
from experiment_management import ExperimentTemplateService, ExperimentLiveInstanceService
import pandas as pd
import textwrap
def wrap(s, width=30):
    return "<br>".join(textwrap.wrap(str(s), width=width))

In [33]:
#getting the data into a single dataframe
base_template = {
    "loss_name": tom.LOSS_NAME,
    "loss_additional_parameters": tom.LOSS_KWARGS,
    "optimizer_name": tom.OPTIMIZER_NAME,
    "optimizer_additional_parameters": tom.OPTIMIZER_KWARGS,
    "model_name": tom.MODEL_ID,
    "module_name": tom.MODULE_NAME,
}
temp_et = ExperimentTemplateService.create_non_persisted(**base_template)
temp_list = ExperimentTemplateService.find_matching(temp_et,excluded = ["group_id","batch_size","normalization","experiment_template_id"])
temp_list = ExperimentTemplateService.find_matching(temp_et,excluded = ["group_id","batch_size","normalization","experiment_template_id"])
#we know they are all same prompt group for now
rows = []
for et in temp_list:
    live_instances = ExperimentLiveInstanceService.find_by({"experiment_template_id":et.experiment_template_id})
    norm = et.normalization
    snapshots = [sorted(live.snapshots,key=lambda snap: snap.iteration_count)[0] for live in live_instances]
    for live,snap in zip(live_instances,snapshots):
        vec_id = live.initial_vector_id
        steered_on_vanilla = list(filter(lambda metric: metric.description == "perplexity_steered_on_vanilla_generated_text",snap.metrics))[0].value
        steered_on_steered = list(filter(lambda metric: metric.description == "perplexity_steered_on_steered_generated_text",snap.metrics))[0].value
        vanilla_on_steered = list(filter(lambda metric: metric.description == "perplexity_vanilla_on_steered_generated_text",snap.generated_outputs[0].metrics))[0].value
        generated_text = wrap(snap.generated_outputs[0].text)
        row = {
            "vector":vec_id,
            "magnitude":norm,
            "steered on vanilla":steered_on_vanilla,
            "steered on steered": steered_on_steered,
            "vanilla on steered": vanilla_on_steered,
            "generated text": generated_text
        }

        if norm < 9:rows.append(row) #remove the high magnitude ones
df = pd.DataFrame(rows)

In [34]:
import plotly.express as px


# df: columns ['vector', 'magnitude', 'steered on vanilla', ...]
fig = px.line(
    df,
    x="magnitude",
    y="steered on vanilla",
    color="vector",
    hover_data=["steered on steered","vanilla on steered","generated text"], 

    markers=True  # optional: show markers
)

fig.update_layout(
    xaxis_title="magnitude",
    yaxis_title="steered on vanilla",
    legend_title="vector"
)

fig.show()

In [41]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

y_cols = ['steered on vanilla','steered on steered','vanilla on steered']

fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=None,
                    subplot_titles=y_cols)

vectors = list(df['vector'].unique())
# ensure exactly 5 colors for 5 vectors
colors = {
    vectors[0]: "#1f77b4",  # blue
    vectors[1]: "#ff7f0e",  # orange
    vectors[2]: "#2ca02c",  # green
    vectors[3]: "#d62728",  # red
    vectors[4]: "#9467bd",  # purple
}

for row, y_col in enumerate(y_cols, start=1):
    for vec in vectors:
        d = df[df['vector'] == vec]
        fig.add_trace(
            go.Scatter(
                x=d['magnitude'],
                y=d[y_col],
                mode='lines+markers',
                name=f"vector {vec}",            # <-- prefix added
                legendgroup=str(vec),
                line=dict(color=colors[vec]),
                marker=dict(color=colors[vec]),
                hovertext = d["generated text"]
            ),
            row=row, col=1
        )

fig.update_layout(height=900, xaxis_title='magnitude')
fig.show(renderer='browser')


Opening in existing browser session.
